In [1]:
from __future__ import annotations
 
import sys
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
BASE_DIR = Path("../data/raw")

datasets = {
    "channels": [],
}

for csv_file in BASE_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(csv_file)

        # df["source_file"] = csv_file.name
        # df["source_path"] = str(csv_file)

        id_dir = next(
            (part for part in csv_file.parts if part.startswith("id_")),
            None
        )
        df["collection_id"] = id_dir

        stem_parts = csv_file.stem.split("_")
        country = stem_parts[-1] if len(stem_parts) > 2 else None
        df["country"] = country

        filename = csv_file.name.lower()


        if "channel" in filename:
            datasets["channels"].append(df)

    except Exception as e:
        print(f"Erro ao ler {csv_file}: {e}")


channels_df = pd.concat(datasets["channels"], ignore_index=True) \
    if datasets["channels"] else pd.DataFrame()

print(f"Channels: {len(channels_df)}")


Channels: 3970


In [2]:
from sklearn.impute import SimpleImputer

df = channels_df.copy()
df = df.drop_duplicates(subset="channel_id")
text_cols = ["channel_id", "title", "description", "custom_url", "country"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].replace(["", "None", "nan", "NaN", "unknown", "UNKNOWN"], np.nan)

if "published_at" in df.columns:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")

num_cols = ["view_count", "subscriber_count", "video_count", "hidden_subscriber_count"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "country" in df.columns:
    df["country"] = (
        df["country"]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({"": np.nan})
    )

if "custom_url" in df.columns:
    df["has_custom_url"] = df["custom_url"].notna().astype(int)
else:
    df["has_custom_url"] = 0

if "hidden_subscriber_count" in df.columns:
    df["hidden_subscriber_count"] = (
        df["hidden_subscriber_count"]
        .fillna(0)
        .astype(int)
        .clip(0, 1)
    )
else:
    df["hidden_subscriber_count"] = 0

for col in ["view_count", "subscriber_count", "video_count"]:
    if col in df.columns:
        df[f"{col}_missing"] = df[col].isna().astype(int)

imputer = SimpleImputer(strategy="median")
cols_to_impute = [c for c in ["view_count", "subscriber_count", "video_count"] if c in df.columns]
if cols_to_impute:
    df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

if "view_count" in df.columns:
    df["view_count_log"] = np.log1p(df["view_count"])
if "subscriber_count" in df.columns:
    df["subscriber_count_log"] = np.log1p(df["subscriber_count"])
if "video_count" in df.columns:
    df["video_count_log"] = np.log1p(df["video_count"])

if set(["subscriber_count", "video_count"]).issubset(df.columns):
    df["subscribers_per_video"] = np.where(
        df["video_count"] > 0,
        df["subscriber_count"] / df["video_count"],
        np.nan
    )

if set(["view_count", "video_count"]).issubset(df.columns):
    df["views_per_video"] = np.where(
        df["video_count"] > 0,
        df["view_count"] / df["video_count"],
        np.nan
    )

if set(["subscriber_count", "view_count"]).issubset(df.columns):
    df["subscribers_per_view"] = np.where(
        df["view_count"] > 0,
        df["subscriber_count"] / df["view_count"],
        np.nan
    )

if "published_at" in df.columns:
    df["channel_year"] = df["published_at"].dt.year
    df["channel_month"] = df["published_at"].dt.month
channel_clean = df[["channel_id","subscriber_count", "view_count", "video_count"]]
channel_clean


,channel_id,subscriber_count,view_count,video_count
0,UCZBY6V8Lxmwu8gGRBOyO11w,6580000.0,3.596700e+09,3194.0
1,UCWVSknuLPIRVGWf-4bFKT4Q,4650000.0,9.121155e+08,869.0
2,UCedicLbmN_7vPUmPXlvKPFg,1540000.0,4.873240e+08,951.0
3,UCNvzD7Z-g64bPXxGzaQaa4g,8600000.0,4.453150e+09,4744.0
4,UCuhwvTYFvVZjyBVXc9T7CrA,3100000.0,8.927644e+08,500.0
...,...,...,...,...
3964,UCH4HIEGhVcH8zvCuZA7J7Sg,225000.0,2.580483e+08,1000.0
3965,UCWcp1Mwm7_bJ-mVoZb8TdkQ,856000.0,2.286402e+08,1467.0
3967,UCpNN2y7AmffcXlgkLUJqNLw,474000.0,6.228448e+08,137.0
3968,UCEgF6KhiOxSe-oRDUJba3Kg,1030000.0,2.112249e+08,1115.0


In [7]:
from pathlib import Path

BASE_DIR = Path("../data/processed")
for csv_file in BASE_DIR.rglob("processed_videos_20260601_153039.csv"):
    
# # Scan the current working directory
# for item in Path('.').iterdir():
#     if item.is_file():
#         print(item.name)
    videos_df = pd.read_csv(csv_file)
videos_df.head()

,video_id,title,channel_id,published_at,category_id,view_count,like_count,comment_count,collection_id,country,engagement_rate,like_ratio,comment_ratio,year,month,view_count_capped
0,Vagb9BqdX8g,We Almost Lost EVERYTHING Gambling,UC-gW4TeZAlKm7UATp24JsWQ,2026-05-31 19:13:22+00:00,20,1016152.0,52805.0,1511.0,id_20,us,0.111366,0.051966,0.001487,2026,5,1016152.0
1,NtFNQEry2X0,PLAYING HARDCORE MINECRAFT UNTIL WE BEAT IT,UCjiXtODGCCulmhwypZAWSag,2026-06-01 06:21:44+00:00,24,1243305.0,16786.0,105.0,id_20,us,0.027424,0.013501,0.000084,2026,6,1243305.0
2,UnFIrVKA9Zc,1000 VS 1000 Player Minecraft Civil War,UCvYPobTo42NM36X7VC4dLhA,2026-06-01 04:10:04+00:00,20,896961.0,79268.0,19726.0,id_20,us,0.286708,0.088374,0.021992,2026,6,896961.0
3,OlQEoEPMJUo,"Hydroneer Tried to Optimize, So I Double Overl...",UCto7D1L-MiRoOziCXK9uT5Q,2026-05-31 14:49:03+00:00,20,1255432.0,72728.0,3050.0,id_20,us,0.128009,0.057931,0.002429,2026,5,1255432.0
4,A8ZfNutzOTM,The Wheel of MLB the Show!,UCaYxyR9mzVlTrOOyZD0XAmA,2026-05-31 22:00:26+00:00,17,170738.0,12163.0,1521.0,id_20,us,0.187018,0.071238,0.008908,2026,5,170738.0


In [ ]:
videos = videos_df.copy()
channels = channel_clean.copy()

if "published_at" in videos.columns:
    videos["published_at"] = pd.to_datetime(videos["published_at"], errors="coerce")

if "published_at" in channels.columns:
    channels = channels.rename(columns={"published_at": "channel_published_at"})

merged = videos.merge(
    channels,
    on="channel_id",
    how="left",
    suffixes=("", "_channel")
)

if set(["published_at", "channel_published_at"]).issubset(merged.columns):
    merged["channel_age_days_at_video"] = (
        merged["published_at"] - merged["channel_published_at"]
    ).dt.days

    merged["channel_age_days_at_video"] = merged["channel_age_days_at_video"].clip(lower=0)

    merged["channel_age_years_at_video"] = merged["channel_age_days_at_video"] / 365.25

if "channel_age_days_at_video" in merged.columns:
    merged["channel_age_days_at_video_missing"] = merged["channel_age_days_at_video"].isna().astype(int)

merged

,video_id,title,channel_id,published_at,category_id,view_count,like_count,comment_count,collection_id,country,engagement_rate,like_ratio,comment_ratio,year,month,view_count_capped,subscriber_count,view_count_channel,video_count
0,Vagb9BqdX8g,We Almost Lost EVERYTHING Gambling,UC-gW4TeZAlKm7UATp24JsWQ,2026-05-31 19:13:22+00:00,20,1016152.0,52805.0,1511.0,id_20,us,0.111366,0.051966,0.001487,2026,5,1016152.0,4330000.0,1.483851e+09,431.0
1,NtFNQEry2X0,PLAYING HARDCORE MINECRAFT UNTIL WE BEAT IT,UCjiXtODGCCulmhwypZAWSag,2026-06-01 06:21:44+00:00,24,1243305.0,16786.0,105.0,id_20,us,0.027424,0.013501,0.000084,2026,6,1243305.0,6500000.0,2.416704e+09,2094.0
2,UnFIrVKA9Zc,1000 VS 1000 Player Minecraft Civil War,UCvYPobTo42NM36X7VC4dLhA,2026-06-01 04:10:04+00:00,20,896961.0,79268.0,19726.0,id_20,us,0.286708,0.088374,0.021992,2026,6,896961.0,2570000.0,2.513280e+08,141.0
3,OlQEoEPMJUo,"Hydroneer Tried to Optimize, So I Double Overl...",UCto7D1L-MiRoOziCXK9uT5Q,2026-05-31 14:49:03+00:00,20,1255432.0,72728.0,3050.0,id_20,us,0.128009,0.057931,0.002429,2026,5,1255432.0,6460000.0,1.733519e+09,749.0
4,A8ZfNutzOTM,The Wheel of MLB the Show!,UCaYxyR9mzVlTrOOyZD0XAmA,2026-05-31 22:00:26+00:00,17,170738.0,12163.0,1521.0,id_20,us,0.187018,0.071238,0.008908,2026,5,170738.0,3210000.0,1.946577e+09,3458.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2067,JmehSrKB-RI,"Introducing the ""VitaWear SmartBand,"" a next-g...",UC-vCSIE1qjHMZqgpPKXT2gQ,2026-05-29 01:00:34+00:00,22,280339.0,9575.0,14.0,id_28,ar,0.068560,0.034155,0.000050,2026,5,280339.0,63200.0,5.738019e+06,68.0
2068,AS8BE51Xp_M,Pintei o celular dos correios #celular #gambia...,UCA1N-DZxhLnccJLmHMB3-7A,2026-05-05 22:15:59+00:00,20,957438.0,45462.0,222.0,id_28,ar,0.096125,0.047483,0.000232,2026,5,957438.0,46000.0,2.850757e+07,295.0
2069,dQ2-f9UseZ8,iPhone 17 Pro Max vs POCO F8 Ultra | La batalla,UCxz7sBKlpcpyCiPwXupRymw,2026-05-30 01:30:06+00:00,28,86979.0,6100.0,859.0,id_28,ar,0.189643,0.070132,0.009876,2026,5,86979.0,4760000.0,1.575329e+09,8391.0
2070,fc1rojR4Rgw,Esta Nueva App Apareció Sola En Tu Móvil ¡Bórr...,UCKYqU0gh6_4Ge2eHO-i-t_A,2026-05-15 17:00:11+00:00,27,765764.0,28950.0,1125.0,id_28,ar,0.082956,0.037805,0.001469,2026,5,765764.0,421000.0,5.514592e+07,503.0
